05 — Model Training
TalentIQ Technologies | ADS508 Data Science with Cloud Computing
Authors: Ramin Fazli & Stephanie Smith
GitHub: https://github.com/raminfazlirf/MLOps-for-Data-Science-ADS-508-02-
Team Repo: https://github.com/stjsmith8/ADS_508_Final_Project

Prepared data source: https://github.com/stjsmith8/ADS_508_Final_Project/tree/main/data/processed

In [34]:
import boto3
import sagemaker
import pandas as pd
import numpy as np

from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
bucket = "ads508-team-4-private"

print("Region:", region)
print("Role:", role)

Region: us-east-1
Role: arn:aws:iam::802896854792:role/LabRole


In [35]:
# load datasets from S3 bucket
train_path = f"s3://{bucket}/processed/train_attrition_original.csv"
val_path   = f"s3://{bucket}/processed/validation_attrition_data.csv"
test_path  = f"s3://{bucket}/processed/test_attrition_data.csv"

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print(train_df.head())

Train shape: (9176, 28)
Validation shape: (1147, 28)
Test shape: (1147, 28)
   Age  DistanceFromHome  JobInvolvement  JobLevel  JobSatisfaction  \
0   54                16               1         3                3   
1   20                21               3         3                2   
2   30                 8               2         1                4   
3   33                 7               3         2                1   
4   40                11               3         2                1   

   EnvironmentSatisfaction  RelationshipSatisfaction  WorkLifeBalance  \
0                        3                         3                4   
1                        1                         3                3   
2                        3                         3                2   
3                        4                         1                3   
4                        1                         1                4   

   MonthlyIncome  NumCompaniesWorked  ...    salary_gap  O

In [36]:
# create scale weight for rebalancing
scale_pos_weight = (train_df["Attrition"] == 0).sum() / (train_df["Attrition"] == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 4.134862898712926


In [42]:
# ensure we only keep wanted features
selected_features = [
    "OverTime_Yes",
    "JobInvolvement",
    "WorkLifeBalance",
    "StagnationScore",
    "NoRecentPromotion",
    "SatisfactionIndex",
    "salary_gap",
    "external_job_demand",
    "TenureCategory_3-5 years",
    "TenureCategory_6-10 years",
    "TenureCategory_10+ years",
    "IncomeCategory_Medium",
    "IncomeCategory_High",
]

selected_features = [c for c in selected_features if c in train_df.columns]

target_col = "Attrition"

train_df = train_df[[target_col] + selected_features]
val_df   = val_df[[target_col] + selected_features]
test_df  = test_df[[target_col] + selected_features]

print("Final feature count:", len(selected_features))
print("Training columns:", train_df.columns.tolist())

Final feature count: 13
Training columns: ['Attrition', 'OverTime_Yes', 'JobInvolvement', 'WorkLifeBalance', 'StagnationScore', 'NoRecentPromotion', 'SatisfactionIndex', 'salary_gap', 'external_job_demand', 'TenureCategory_3-5 years', 'TenureCategory_6-10 years', 'TenureCategory_10+ years', 'IncomeCategory_Medium', 'IncomeCategory_High']


In [43]:
# update format so we can use Sagemaker built in XGboost
train_xgb_local = "/tmp/train_xgb.csv"
val_xgb_local   = "/tmp/validation_xgb.csv"
test_xgb_local  = "/tmp/test_xgb.csv"

train_df.to_csv(train_xgb_local, index=False, header=False)
val_df.to_csv(val_xgb_local, index=False, header=False)
test_df.to_csv(test_xgb_local, index=False, header=False)

print("Saved local built-in XGBoost files.")

Saved local built-in XGBoost files.


In [44]:
#upload to S3
train_xgb_s3 = f"s3://{bucket}/processed_builtin/train/train_xgb.csv"
val_xgb_s3   = f"s3://{bucket}/processed_builtin/validation/validation_xgb.csv"
test_xgb_s3  = f"s3://{bucket}/processed_builtin/test/test_xgb.csv"

session.upload_data(train_xgb_local, bucket=bucket, key_prefix="processed_builtin/train")
session.upload_data(val_xgb_local, bucket=bucket, key_prefix="processed_builtin/validation")
session.upload_data(test_xgb_local, bucket=bucket, key_prefix="processed_builtin/test")

print(train_xgb_s3)
print(val_xgb_s3)
print(test_xgb_s3)

s3://ads508-team-4-private/processed_builtin/train/train_xgb.csv
s3://ads508-team-4-private/processed_builtin/validation/validation_xgb.csv
s3://ads508-team-4-private/processed_builtin/test/test_xgb.csv


In [45]:
# get built in xgboost container
xgb_image = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1"
)

print("XGBoost image:", xgb_image)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


XGBoost image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


In [46]:
# create Sagemaker training job
xgb_estimator = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/model-artifacts/",
    sagemaker_session=session
)

xgb_estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    num_round=300,
    max_depth=4,
    eta=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.05,
    alpha=0.1,
    scale_pos_weight=scale_pos_weight,
    **{"lambda": 1.0}
)

train_input = TrainingInput(
    s3_data=f"s3://{bucket}/processed_builtin/train/",
    content_type="text/csv"
)

validation_input = TrainingInput(
    s3_data=f"s3://{bucket}/processed_builtin/validation/",
    content_type="text/csv"
)

xgb_estimator.fit({
    "train": train_input,
    "validation": validation_input
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-04-11-20-53-40-848


2026-04-11 20:53:43 Starting - Starting the training job...
2026-04-11 20:53:57 Starting - Preparing the instances for training...
2026-04-11 20:54:46 Downloading - Downloading the training image......
2026-04-11 20:55:37 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-04-11 20:55:47.901 ip-10-2-92-66.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-04-11 20:55:47.975 ip-10-2-92-66.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-04-11:20:55:48:INFO] Imported framework sagemaker_xgboost_container.training
[2026-04-11:20:55:48:INFO] Failed to parse hyperparameter ev

In [47]:
#run hyperparameter tuning through Sagemaker
hyperparameter_ranges = {
    "max_depth": IntegerParameter(3, 8),
    "eta": ContinuousParameter(0.01, 0.3),
    "subsample": ContinuousParameter(0.5, 1.0),
    "colsample_bytree": ContinuousParameter(0.5, 1.0),
    "min_child_weight": IntegerParameter(1, 10),
    "gamma": ContinuousParameter(0.0, 5.0),
    "alpha": ContinuousParameter(0.0, 5.0),
    "lambda": ContinuousParameter(0.0, 5.0),
    "num_round": IntegerParameter(100, 500),
}

tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name="validation:auc",
    hyperparameter_ranges=hyperparameter_ranges,
    objective_type="Maximize",
    max_jobs=9,
    max_parallel_jobs=3,
    strategy="Bayesian"
)

In [48]:
# run tuning 
tuner.fit({
    "train": train_input,
    "validation": validation_input
}, wait=True)

INFO:sagemaker:Creating hyperparameter tuning job with name: sagemaker-xgboost-260411-2056


......................................................!


In [49]:
# get the best training job and best estimators
best_job_name = tuner.best_training_job()
print("Best training job:", best_job_name)

best_estimator = tuner.best_estimator()
print("Best estimator model data:", best_estimator.model_data)

Best training job: sagemaker-xgboost-260411-2056-005-38086c4e

2026-04-11 21:00:27 Starting - Found matching resource for reuse
2026-04-11 21:00:27 Downloading - Downloading the training image
2026-04-11 21:00:27 Training - Training image download completed. Training in progress.
2026-04-11 21:00:27 Uploading - Uploading generated training model
2026-04-11 21:00:27 Completed - Resource reused by training job: sagemaker-xgboost-260411-2056-009-5ed79c67
Best estimator model data: s3://ads508-team-4-private/model-artifacts/sagemaker-xgboost-260411-2056-005-38086c4e/output/model.tar.gz
